# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhaibachaopls-web/Flyrani_repo/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
%pip -q install duckdb huggingface_hub


In [6]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


In [7]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Feature Vector Construction Strategy:

Engineered Features: Backward-looking rolling windows for volume (past_7d_impressions, past_7d_clicks), engagement rates (past_7d_ctr), and organic search rank (past_7d_avg_pos).

Categorical Handling: Binary flags converted to numerical integer indicators (has_ga4_tracking).

Null Fills: COALESCE replaces missing impression/click counts with 0, missing CTR with 0.0, and missing position metrics with a default unranked baseline (100.0).

Target Label: future_7d_clicks accumulated across strictly future days (+1 to +7 days).

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


feature_vector_df = con.sql(f"""
    WITH daily_base AS (
        SELECT
            content_hash_id,
            client_hash_id,
            report_date,
            COALESCE(gsc_impressions, 0) AS gsc_impressions,
            COALESCE(gsc_clicks, 0) AS gsc_clicks,
            COALESCE(gsc_avg_position, 100.0) AS gsc_avg_position,
            COALESCE(ga4_data_available, FALSE) AS ga4_data_available
        FROM {TABLES['fact_daily']}
        WHERE report_date >= '2026-02-20' AND report_date <= '2026-03-31'
          AND gsc_data_available IS TRUE
    ),
    windowed AS (
        SELECT
            content_hash_id,
            client_hash_id,
            report_date,

            -- Past 7-Day Rolling Aggregations (Strictly Historical)
            SUM(gsc_impressions) OVER (
                PARTITION BY content_hash_id
                ORDER BY report_date
                ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
            ) AS past_7d_impressions,

            SUM(gsc_clicks) OVER (
                PARTITION BY content_hash_id
                ORDER BY report_date
                ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
            ) AS past_7d_clicks,

            AVG(gsc_avg_position) OVER (
                PARTITION BY content_hash_id
                ORDER BY report_date
                ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
            ) AS past_7d_avg_pos,

            -- Categorical Handling
            CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END AS has_ga4_tracking,

            -- Future Label (Strictly Forward-Looking)
            SUM(gsc_clicks) OVER (
                PARTITION BY content_hash_id
                ORDER BY report_date
                ROWS BETWEEN 1 FOLLOWING AND 7 FOLLOWING
            ) AS future_7d_clicks

        FROM daily_base
    )
    SELECT
        content_hash_id,
        client_hash_id,
        report_date,
        COALESCE(past_7d_impressions, 0) AS past_7d_impressions,
        COALESCE(past_7d_clicks, 0) AS past_7d_clicks,
        COALESCE(past_7d_clicks / NULLIF(past_7d_impressions, 0), 0.0) AS past_7d_ctr,
        COALESCE(past_7d_avg_pos, 100.0) AS past_7d_avg_pos,
        has_ga4_tracking,
        COALESCE(future_7d_clicks, 0) AS future_7d_clicks
    FROM windowed
    WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-24'
""").df()

print("Feature Vector Shape:", feature_vector_df.shape)
display(feature_vector_df.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature Vector Shape: (2736046, 9)


,content_hash_id,client_hash_id,report_date,past_7d_impressions,past_7d_clicks,past_7d_ctr,past_7d_avg_pos,has_ga4_tracking,future_7d_clicks
0,content_8a7c62650134b1ad,client_e5c2aa26a8598242,2026-03-01,341.0,0.0,0.000000,1.502138,0,1.0
1,content_8a7c62650134b1ad,client_e5c2aa26a8598242,2026-03-02,401.0,1.0,0.002494,1.394690,0,1.0
2,content_8a7c62650134b1ad,client_e5c2aa26a8598242,2026-03-03,469.0,1.0,0.002132,1.263276,0,1.0
3,content_8a7c62650134b1ad,client_e5c2aa26a8598242,2026-03-04,499.0,1.0,0.002004,1.239100,0,1.0
4,content_8a7c62650134b1ad,client_e5c2aa26a8598242,2026-03-05,501.0,1.0,0.001996,1.064894,0,0.0
5,content_8a7c62650134b1ad,client_e5c2aa26a8598242,2026-03-06,473.0,2.0,0.004228,0.939326,0,0.0
6,content_8a7c62650134b1ad,client_e5c2aa26a8598242,2026-03-07,466.0,2.0,0.004292,0.904200,0,2.0
7,content_8a7c62650134b1ad,client_e5c2aa26a8598242,2026-03-08,446.0,2.0,0.004484,0.868138,0,2.0
8,content_8a7c62650134b1ad,client_e5c2aa26a8598242,2026-03-09,452.0,1.0,0.002212,0.854069,0,2.0
9,content_8a7c62650134b1ad,client_e5c2aa26a8598242,2026-03-10,422.0,1.0,0.002370,0.828737,0,2.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Feature Notes:

past_7d_impressions: Total search impressions over the strictly preceding 7 days. Missing values are coerced to 0 (assuming no traffic). It exists BEFORE the prediction moment because the window stops at 1 PRECEDING.

past_7d_clicks: Total search clicks over the strictly preceding 7 days. Missing values are coerced to 0. It exists BEFORE the prediction moment.

past_7d_ctr: The ratio of clicks to impressions over the preceding 7 days. Missing values (and divide-by-zero errors) are coerced to 0.0. It exists BEFORE the prediction moment.

past_7d_avg_pos: The average organic search rank over the preceding 7 days. Missing values are replaced with a penalty value of 100.0 (indicating unranked/invisible). It exists BEFORE the prediction moment.

has_ga4_tracking: A numerical indicator (1/0) of whether the client had Google Analytics 4 connected. Missing values assume 0 (False). This state is known at the time of prediction.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


print("--- Data Types ---")
print(feature_vector_df.dtypes)

print("\n--- Missing Value Count (Should all be 0) ---")
print(feature_vector_df.isnull().sum())

print("\n--- Max CTR Check ---")
print(f"Highest CTR: {feature_vector_df['past_7d_ctr'].max():.4f}")


--- Data Types ---
content_hash_id                object
client_hash_id                 object
report_date            datetime64[us]
past_7d_impressions           float64
past_7d_clicks                float64
past_7d_ctr                   float64
past_7d_avg_pos               float64
has_ga4_tracking                int32
future_7d_clicks              float64
dtype: object

--- Missing Value Count (Should all be 0) ---
content_hash_id        0
client_hash_id         0
report_date            0
past_7d_impressions    0
past_7d_clicks         0
past_7d_ctr            0
past_7d_avg_pos        0
has_ga4_tracking       0
future_7d_clicks       0
dtype: int64

--- Max CTR Check ---
Highest CTR: 1.0000


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

The Leakage Hunt:
To prove our features are honest (strictly knowable at the time of prediction), we will attack them in two ways:

Correlation Check: We will calculate the Pearson correlation of all features against the target label (future_7d_clicks). Any feature with an abnormally high correlation (e.g., > 0.90) is an immediate red flag for label derivation.

Baseline Model Test: We will train a simple Random Forest on our feature vector. If the R-squared score approaches 1.0 (perfect prediction), it means a feature is acting as a "cheat sheet" and leaking the future. A realistic score validates that our PRECEDING window boundaries are secure.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

correlations = feature_vector_df.corr(numeric_only=True)['future_7d_clicks'].sort_values(ascending=False)
display(correlations.to_frame())


features = ['past_7d_impressions', 'past_7d_clicks', 'past_7d_ctr', 'past_7d_avg_pos', 'has_ga4_tracking']
target = 'future_7d_clicks'


clean_df = feature_vector_df.dropna(subset=features + [target])

X = clean_df[features]
y = clean_df[target]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


rf = RandomForestRegressor(n_estimators=15, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
preds = rf.predict(X_test)


r2 = r2_score(y_test, preds)
print(f"R-squared Score: {r2:.4f}")

if r2 > 0.90:
    print("This might be a leaky feature!")
else:
    print("Score seems realistic.")


,future_7d_clicks
future_7d_clicks,1.000000
past_7d_clicks,0.876795
past_7d_impressions,0.570762
has_ga4_tracking,0.159338
past_7d_ctr,0.053429
past_7d_avg_pos,-0.079921


R-squared Score: 0.7858
Score seems realistic.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Fields I Excluded and Why:

month: Ignored because it is a redundant partition column; we already extract strict time windows using report_date.

gsc_sum_position: Refused to use because it is heavily skewed by impression volume. The normalized gsc_avg_position is a much safer indicator of actual organic rank.

GA4 Engagement Metrics (ga4_pageviews, sessions_*, ai_*): Excluded from the feature vector because we proved in our data contract that there is a massive tracking blind spot. Relying on these would force us to drop a huge portion of our GSC-verified dataset or introduce heavy 0-fill bias for clients without GA4 installed.

Current-day raw metrics: Refused to use raw gsc_clicks or gsc_impressions for the current row's date to strictly prevent leakage; everything must be windowed to PRECEDING days.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

raw_columns = con.sql(f"SELECT * FROM {TABLES['fact_daily']} LIMIT 1").df().columns.tolist()
kept_columns = feature_vector_df.columns.tolist()

excluded_columns = set(raw_columns) - set(kept_columns)

print(f"Total raw columns available: {len(raw_columns)}")
print(f"Columns kept for modeling:   {len(kept_columns)}")
print(f"Columns strictly excluded:   {len(excluded_columns)}\n")

print("====================")
for col in sorted(excluded_columns):
    print(f"- {col}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total raw columns available: 31
Columns kept for modeling:   9
Columns strictly excluded:   28

- ai_chatgpt
- ai_claude
- ai_copilot
- ai_gemini
- ai_meta
- ai_other
- ai_perplexity
- client_has_ga4
- client_has_gsc
- ga4_data_available
- ga4_engaged_sessions
- ga4_pageviews
- ga4_sessions
- ga4_total_engagement_sec
- ga4_users
- gsc_avg_position
- gsc_clicks
- gsc_data_available
- gsc_impressions
- gsc_sum_position
- month
- scroll_events
- sessions_ai
- sessions_direct
- sessions_organic
- sessions_paid
- sessions_referral
- sessions_social


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.